In [1]:
%pip install jax optax chex distrax flax navix pathlib numpy wandb typing pillow

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for typing: filename=typing-3.7.4.3-py3-none-any.whl size=26397 sha256=0a85fbe452b8bd2e566d3cf9852ba87e6e7c6cdbc333e62a677ff7e6d562b779
  Stored in directory: /home/bezin/.cache/pip/wheels/12/98/52/2bffe242a9a487f00886e43b8ed8dac46456702e11a0d6abef
Successfully built typing
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [typing]
Note: you may need to restart the kernel to use updated packages.


In [2]:
import jax
import jax.numpy as jnp
import optax
import chex
import distrax
import flax.linen as nn
from flax.training.train_state import TrainState

import navix as nx
from navix import observations

from pathlib import Path

import numpy as np
import wandb
from typing import Tuple, Any, Dict, Sequence, Callable
from PIL import Image 
import functools



In [3]:
# 1. Data Structures: Meticulous shape tracking
# Using chex helps enforce accuracy by allowing us to assert shapes later.
@chex.dataclass
class Transition:
    global_obs: chex.Array  # For the Critic (CTDE)
    symbolic_obs: chex.Array # For the Seer
    local_obs: chex.Array   # For the Doer
    proprioception: chex.Array # For the Doer
    message: chex.Array     # For CIC and Heatmap logging
    doer_action: chex.Array
    doer_log_prob: chex.Array
    seer_action: chex.Array
    seer_log_prob: chex.Array
    value: chex.Array
    reward: chex.Array
    task_reward: chex.Array
    progress_reward: chex.Array
    follow_reward: chex.Array
    cic_reward_component: chex.Array
    cic_score: chex.Array
    step_penalty_component: chex.Array
    bump_penalty_component: chex.Array
    done: chex.Array
    advantage: chex.Array
    return_val: chex.Array


def _build_message_codebook(message_levels: Tuple[int, ...], dtype: jnp.dtype) -> jnp.ndarray:
    axes = [jnp.arange(level, dtype=dtype) for level in message_levels]
    mesh = jnp.meshgrid(*axes, indexing="ij")
    return jnp.stack(mesh, axis=-1).reshape((-1, len(message_levels)))


def _compute_message_entropy_metrics(
    discrete_messages: chex.Array,
    message_levels: Tuple[int, ...],
) -> Tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray]:
    """Approximate discrete-code usage with a soft histogram so gradients can flow."""
    flat_messages = discrete_messages.reshape((-1, discrete_messages.shape[-1]))
    codebook = _build_message_codebook(message_levels, flat_messages.dtype)
    sq_distances = jnp.sum(
        jnp.square(flat_messages[:, None, :] - codebook[None, :, :]),
        axis=-1,
    )
    assignment_probs = nn.softmax(-10.0 * sq_distances, axis=-1)
    code_probs = assignment_probs.mean(axis=0)
    code_probs = code_probs / (code_probs.sum() + 1e-8)
    message_entropy = -jnp.sum(code_probs * jnp.log(code_probs + 1e-8))

    num_codes = codebook.shape[0]
    if num_codes > 1:
        normalized_entropy = message_entropy / jnp.log(jnp.asarray(num_codes, dtype=flat_messages.dtype))
    else:
        normalized_entropy = jnp.asarray(0.0, dtype=flat_messages.dtype)

    dominant_code_prob = jnp.max(code_probs)
    return message_entropy, normalized_entropy, dominant_code_prob

# 2. Loss Functions: Separated from network definitions
def calculate_actor_losses(
    seer_apply_fn: Callable,
    doer_apply_fn: Callable,
    actor_params: dict, # Changed from Any to dict
    transition_batch: Transition,
    init_seer_carry: Tuple[jnp.ndarray, jnp.ndarray], # Changed from Any to Tuple
    init_doer_carry: Tuple[jnp.ndarray, jnp.ndarray], # Changed from Any to Tuple
    control_mode: jnp.ndarray,
    message_levels: Tuple[int, ...],
    clip_eps: float = 0.2,
    entropy_coef: float = 0.01,
    seer_entropy_coef: jnp.ndarray = jnp.array(0.01)
) -> Tuple[jnp.ndarray, dict]:
    """Calculates separate PPO losses for the Seer and Doer reward streams."""
    
    # We must assert shape to prevent silent broadcasting bugs.
    # transition_batch has shape (batch_size, sequence_length, ...) or just (sequence_length, ...)
    # Wait, loop.py returns (num_steps, ...) for single env. Let's assume unbatched or reshape-able.
    # If the user is unrolling over dimension 0:
    
    def scan_fn(carry, transition_step):
        # Step shape assertions:
        # chex.assert_rank(transition_step.action, 1) # (batch_size,)
        # chex.assert_rank(transition_step.global_obs, 4) # (batch_size, H, W, C)
        
        seer_carry, doer_carry = carry
        
        # Seer Forward Pass
        next_seer_carry, discrete_message, thought_vector, seer_nav_logits = seer_apply_fn(
            {"params": actor_params["seer"]},
            seer_carry,
            transition_step.global_obs,
            transition_step.symbolic_obs,
        )
        
        # Doer Forward Pass
        next_doer_carry, logits = doer_apply_fn(
            {"params": actor_params["doer"]},
            doer_carry,
            transition_step.local_obs,
            transition_step.proprioception,
            discrete_message
        )
        return (next_seer_carry, next_doer_carry), (
            logits,
            seer_nav_logits,
            discrete_message,
            thought_vector,
        )
        
    _, (doer_logits, seer_nav_logits, discrete_messages, thought_vectors) = jax.lax.scan(
        scan_fn, 
        (init_seer_carry, init_doer_carry), 
        transition_batch
    )

    communication_mode = control_mode == 1
    doer_pi = distrax.Categorical(logits=doer_logits)
    seer_pi = distrax.Categorical(logits=seer_nav_logits)
    doer_new_log_probs = doer_pi.log_prob(transition_batch.doer_action)
    seer_nav_new_log_probs = seer_pi.log_prob(transition_batch.seer_action)
    doer_entropy = doer_pi.entropy().mean()
    seer_nav_entropy = seer_pi.entropy().mean()

    seer_adv = transition_batch.advantage[..., 0]
    doer_adv = transition_batch.advantage[..., 1]
    seer_adv = (seer_adv - seer_adv.mean()) / (seer_adv.std() + 1e-8)
    doer_adv = (doer_adv - doer_adv.mean()) / (doer_adv.std() + 1e-8)

    seer_old_log_probs = jnp.where(
        communication_mode,
        transition_batch.doer_log_prob,
        transition_batch.seer_log_prob,
    )
    seer_new_log_probs = jnp.where(
        communication_mode,
        doer_new_log_probs,
        seer_nav_new_log_probs,
    )
    seer_logratio = seer_new_log_probs - seer_old_log_probs
    seer_ratio = jnp.exp(seer_logratio)

    seer_loss_unclipped = seer_ratio * seer_adv
    seer_loss_clipped = jnp.clip(seer_ratio, 1.0 - clip_eps, 1.0 + clip_eps) * seer_adv
    seer_actor_loss = -jnp.minimum(seer_loss_unclipped, seer_loss_clipped).mean()

    doer_logratio = doer_new_log_probs - transition_batch.doer_log_prob
    doer_ratio = jnp.exp(doer_logratio)
    doer_loss_unclipped = doer_ratio * doer_adv
    doer_loss_clipped = jnp.clip(doer_ratio, 1.0 - clip_eps, 1.0 + clip_eps) * doer_adv
    doer_actor_loss = -jnp.minimum(doer_loss_unclipped, doer_loss_clipped).mean()
    doer_actor_loss = jnp.where(communication_mode, doer_actor_loss, 0.0)

    message_entropy, message_entropy_normalized, dominant_code_prob = (
        _compute_message_entropy_metrics(discrete_messages, message_levels)
    )
    seer_bonus = jnp.where(communication_mode, message_entropy, seer_nav_entropy)

    seer_loss = seer_actor_loss - seer_entropy_coef * seer_bonus
    doer_loss = doer_actor_loss - jnp.where(communication_mode, entropy_coef * doer_entropy, 0.0)

    return (seer_loss, doer_loss), {
        "seer_actor_loss": seer_actor_loss,
        "doer_actor_loss": doer_actor_loss,
        "entropy": jnp.where(communication_mode, doer_entropy, seer_nav_entropy),
        "seer_ratio": seer_ratio.mean(),
        "doer_ratio": doer_ratio.mean(),
        "message_entropy": message_entropy,
        "message_entropy_normalized": message_entropy_normalized,
        "message_dominant_probability": dominant_code_prob,
        "seer_nav_entropy": seer_nav_entropy,
        "discrete_messages": discrete_messages
    }

def calculate_critic_loss(
    critic_apply_fn: Callable,
    critic_params: Any,
    transition_batch: Transition,
    value_clip: float = 0.2
) -> Tuple[jnp.ndarray, dict]:
    """Calculates the value loss for the centralized critic."""
    
    # Assert shape
    chex.assert_rank(transition_batch.global_obs, 4) # (batch_size, H, W, C) since it is flattened over sequence
    
    # The critic uses the global observation (CTDE)
    values = critic_apply_fn({"params": critic_params}, transition_batch.global_obs)

    value_pred_clipped = transition_batch.value + jnp.clip(
        values - transition_batch.value, -value_clip, value_clip
    )
    
    value_losses = jnp.square(values - transition_batch.return_val)
    value_losses_clipped = jnp.square(value_pred_clipped - transition_batch.return_val)
    
    critic_loss = 0.5 * jnp.maximum(value_losses, value_losses_clipped).mean()
    
    return critic_loss, {"critic_loss": critic_loss}

# 3. The Update Step: JIT-compiled gradient application
import functools

@functools.partial(
    jax.jit,
    static_argnames=(
        "seer_apply_fn",
        "doer_apply_fn",
        "message_levels",
        "num_ppo_epochs",
        "num_minibatches",
    ),
)
def update_actor(
    actor_state: TrainState, 
    transition_batch: Transition,
    init_seer_carry: Any,
    init_doer_carry: Any,
    seer_apply_fn: Callable,
    doer_apply_fn: Callable,
    rng: jax.random.PRNGKey,
    control_mode: jnp.ndarray,
    message_levels: Tuple[int, ...],
    seer_entropy_coef: jnp.ndarray,
    num_ppo_epochs: int = 4,
    num_minibatches: int = 1
) -> Tuple[TrainState, dict]:
    """Computes gradients and updates the actor network using PPO epochs."""
    
    # Add a batch dimension if missing (assumes trajectory is num_steps, ...)
    # Wait, the prompt says trajectory is (batch_size, seq_len, ...)
    # If the user scales num_envs later, batch_size = num_envs
    
    batch_size = transition_batch.doer_action.shape[0]
    minibatch_size = batch_size // num_minibatches
    
    def epoch_fn(carry, _):
        actor_state, key = carry
        key, subkey = jax.random.split(key)
        
        # Shuffle along the batch dimension
        permutation = jax.random.permutation(subkey, batch_size)
        
        def minibatch_fn(state, start_idx):
            # Slice the minibatch
            indices = jax.lax.dynamic_slice_in_dim(permutation, start_idx, minibatch_size)
            mb_transition = jax.tree_util.tree_map(lambda x: x[indices], transition_batch)
            
            # Since calculate_actor_loss currently assumes scan over time, and time is dim 1:
            # wait, if input is (batch, time, ...) and scan is over time, we must swap axes!
            # scan_fn expects transition sequence to be the leading dimension.
            # So let's swap seq_len (dim 1) to be dim 0 for scan.
            mb_transition_time_first = jax.tree_util.tree_map(lambda x: jnp.swapaxes(x, 0, 1), mb_transition)
            
            # Extract initial carries for this minibatch
            # Assuming init_seer_carry is (batch_size, ...)
            mb_seer_carry = jax.tree_util.tree_map(lambda x: x[indices], init_seer_carry)
            mb_doer_carry = jax.tree_util.tree_map(lambda x: x[indices], init_doer_carry)
            
            def seer_loss_fn(seer_params):
                (seer_loss, _), metrics = calculate_actor_losses(
                    seer_apply_fn,
                    doer_apply_fn,
                    {"seer": seer_params, "doer": state.params["doer"]},
                    mb_transition_time_first,
                    mb_seer_carry,
                    mb_doer_carry,
                    control_mode=control_mode,
                    message_levels=message_levels,
                    seer_entropy_coef=seer_entropy_coef,
                )
                return seer_loss, metrics

            def doer_loss_fn(doer_params):
                (_, doer_loss), metrics = calculate_actor_losses(
                    seer_apply_fn,
                    doer_apply_fn,
                    {"seer": state.params["seer"], "doer": doer_params},
                    mb_transition_time_first,
                    mb_seer_carry,
                    mb_doer_carry,
                    control_mode=control_mode,
                    message_levels=message_levels,
                    seer_entropy_coef=seer_entropy_coef,
                )
                return doer_loss, metrics

            (seer_loss, seer_metrics), seer_grads = jax.value_and_grad(
                seer_loss_fn,
                has_aux=True,
            )(state.params["seer"])
            (doer_loss, doer_metrics), doer_grads = jax.value_and_grad(
                doer_loss_fn,
                has_aux=True,
            )(state.params["doer"])

            grads = {"seer": seer_grads, "doer": doer_grads}

            # Record explicit gradient norms for auditing
            seer_grad_norm = optax.global_norm(grads["seer"])
            doer_grad_norm = optax.global_norm(grads["doer"])

            metrics = {
                "seer_loss": seer_loss,
                "doer_loss": doer_loss,
                "seer_actor_loss": seer_metrics["seer_actor_loss"],
                "doer_actor_loss": doer_metrics["doer_actor_loss"],
                "entropy": doer_metrics["entropy"],
                "seer_ratio": seer_metrics["seer_ratio"],
                "doer_ratio": doer_metrics["doer_ratio"],
                "message_entropy": seer_metrics["message_entropy"],
                "message_entropy_normalized": seer_metrics["message_entropy_normalized"],
                "message_dominant_probability": seer_metrics["message_dominant_probability"],
                "seer_nav_entropy": seer_metrics["seer_nav_entropy"],
                "discrete_messages": seer_metrics["discrete_messages"],
                "seer_grad_norm": seer_grad_norm,
                "doer_grad_norm": doer_grad_norm,
                "actor_loss": seer_loss + doer_loss,
            }
            
            new_state = state.apply_gradients(grads=grads)
            return new_state, metrics
            
        # Minibatch loop (scan over start_indices)
        start_indices = jnp.arange(0, batch_size, minibatch_size)
        actor_state, mb_metrics = jax.lax.scan(minibatch_fn, actor_state, start_indices)
        
        # Average metrics over minibatches
        epoch_metrics = {k: v.mean() if k != "discrete_messages" else v[0] for k, v in mb_metrics.items()}
        return (actor_state, key), epoch_metrics
        
    (final_actor_state, _), epoch_metrics = jax.lax.scan(
        epoch_fn, (actor_state, rng), None, length=num_ppo_epochs
    )
    
    # Return averaged metrics over epochs
    final_metrics = {k: v.mean() if k != "discrete_messages" else v[0] for k, v in epoch_metrics.items()}
    return final_actor_state, final_metrics

@functools.partial(jax.jit, static_argnames=("critic_apply_fn", "num_ppo_epochs", "num_minibatches"))
def update_critic(
    critic_state: TrainState, 
    transition_batch: Transition,
    critic_apply_fn: Callable,
    rng: jax.random.PRNGKey,
    num_ppo_epochs: int = 4,
    num_minibatches: int = 1
) -> Tuple[TrainState, dict]:
    """Computes gradients and updates the critic network."""
    
    # For critic we can flatten the batch and sequence dimension since it's just MLPs
    batch_size = transition_batch.doer_action.shape[0] * transition_batch.doer_action.shape[1]
    flat_transition = jax.tree_util.tree_map(lambda x: x.reshape((batch_size,) + x.shape[2:]), transition_batch)
    minibatch_size = batch_size // num_minibatches
    
    def epoch_fn(carry, _):
        critic_state, key = carry
        key, subkey = jax.random.split(key)
        permutation = jax.random.permutation(subkey, batch_size)
        
        def minibatch_fn(state, start_idx):
            indices = jax.lax.dynamic_slice_in_dim(permutation, start_idx, minibatch_size)
            mb_transition = jax.tree_util.tree_map(lambda x: x[indices], flat_transition)
            
            loss_fn = lambda params: calculate_critic_loss(
                critic_apply_fn, params, mb_transition
            )
            
            (loss, metrics), grads = jax.value_and_grad(loss_fn, has_aux=True)(state.params)
            new_state = state.apply_gradients(grads=grads)
            return new_state, metrics
            
        start_indices = jnp.arange(0, batch_size, minibatch_size)
        critic_state, mb_metrics = jax.lax.scan(minibatch_fn, critic_state, start_indices)
        
        epoch_metrics = jax.tree_util.tree_map(lambda x: x.mean(), mb_metrics)
        return (critic_state, key), epoch_metrics

    (final_critic_state, _), epoch_metrics = jax.lax.scan(
        epoch_fn, (critic_state, rng), None, length=num_ppo_epochs
    )
    
    final_metrics = jax.tree_util.tree_map(lambda x: x.mean(), epoch_metrics)
    return final_critic_state, final_metrics


In [4]:
UNSET_POSITION = jnp.array([-1, -1], dtype=jnp.int32)


class NavixGridWrapper:
    """Expose Navix single-agent environments with the Seer/Doer observation split."""

    SEER_NAV_PHASE = 0
    COMMUNICATION_PHASE = 1

    def __init__(
        self,
        env,
        progress_reward_scale: float = 0.1,
        min_start_distance: float = 0.0,
        step_penalty: float = 0.0,
        bump_penalty: float = 0.1,
        doer_perception_level: int = 0,
    ):
        self._env = env
        self.progress_reward_scale = progress_reward_scale
        self.min_start_distance = jnp.asarray(min_start_distance, dtype=jnp.float32)
        self.step_penalty = jnp.asarray(step_penalty, dtype=jnp.float32)
        self.bump_penalty = jnp.asarray(bump_penalty, dtype=jnp.float32)
        self.doer_perception_level = int(doer_perception_level)

    @property
    def num_actions(self) -> int:
        # Navigation-only action space: turn left, turn right, move forward.
        return 3

    @staticmethod
    def player_position(timestep) -> jnp.ndarray:
        return timestep.state.get_player().position.astype(jnp.int32)

    @staticmethod
    def goal_position(timestep) -> jnp.ndarray:
        return timestep.state.get_goals().position[0].astype(jnp.int32)

    @staticmethod
    def _position_matches(actual_position: jnp.ndarray, target_position: jnp.ndarray) -> jnp.ndarray:
        return jnp.logical_or(
            jnp.any(target_position < 0),
            jnp.all(actual_position == target_position),
        )

    def _apply_curriculum_positions(
        self,
        timestep,
        fixed_goal_position: jnp.ndarray,
        fixed_start_position: jnp.ndarray,
    ):
        def set_goal(state):
            goals = state.get_goals()
            updated_goal_positions = goals.position.at[0].set(
                fixed_goal_position.astype(goals.position.dtype)
            )
            return state.set_goals(goals.replace(position=updated_goal_positions))

        def set_start(state):
            player = state.get_player()
            return state.set_player(
                player.replace(
                    position=fixed_start_position.astype(player.position.dtype)
                )
            )

        state = jax.lax.cond(
            jnp.all(fixed_goal_position >= 0),
            set_goal,
            lambda state: state,
            timestep.state,
        )
        state = jax.lax.cond(
            jnp.all(fixed_start_position >= 0),
            set_start,
            lambda state: state,
            state,
        )

        return timestep.replace(state=state)

    def _split_observations(
        self,
        timestep,
        vision_radius: jnp.ndarray,
        control_mode: jnp.ndarray,
    ):
        state = timestep.state
        global_map = observations.symbolic(state).astype(jnp.float32)
        full_local_view = observations.symbolic_first_person(state).astype(jnp.float32)

        player = state.get_player()
        goal = state.get_goals()
        center_row = full_local_view.shape[0] // 2
        center_col = full_local_view.shape[1] // 2
        local_view_3x3 = jax.lax.dynamic_slice(
            full_local_view,
            (center_row - 1, center_col - 1, 0),
            (3, 3, full_local_view.shape[-1]),
        )

        symbolic_state = jnp.array(
            [
                player.position[0],
                player.position[1],
                player.direction,
                player.pocket,
                goal.position[0, 0],
                goal.position[0, 1],
                (control_mode == self.SEER_NAV_PHASE).astype(jnp.float32),
            ],
            dtype=jnp.float32,
        )

        proprioception_full = jnp.array(
            [
                player.position[0],
                player.position[1],
                player.direction,
                player.pocket,
            ],
            dtype=jnp.float32,
        )

        if self.doer_perception_level == 0:
            local_view = local_view_3x3
            proprioception = proprioception_full
        elif self.doer_perception_level == 1:
            local_view = jnp.zeros_like(local_view_3x3)
            local_view = local_view.at[0, 1].set(local_view_3x3[0, 1])
            proprioception = proprioception_full
        elif self.doer_perception_level == 2:
            local_view = jnp.zeros_like(local_view_3x3)
            proprioception = jnp.array(
                [
                    player.position[0],
                    player.position[1],
                    0.0,
                    0.0,
                ],
                dtype=jnp.float32,
            )
        elif self.doer_perception_level == 3:
            local_view = jnp.zeros_like(local_view_3x3)
            proprioception = jnp.zeros_like(proprioception_full)
        else:
            raise ValueError(
                f"Unsupported doer_perception_level={self.doer_perception_level}. "
                "Use 0, 1, 2, or 3."
            )

        return {
            "global_map": global_map,
            "symbolic_state": symbolic_state,
            "local_view": local_view,
            "proprioception": proprioception,
        }

    @staticmethod
    def _goal_distance(state) -> jnp.ndarray:
        player = state.get_player()
        goal = state.get_goals()
        return jnp.abs(player.position - goal.position[0]).sum().astype(jnp.float32)

    def reset(
        self,
        key: jnp.ndarray,
        vision_radius: jnp.ndarray = jnp.array(3.0),
        control_mode: jnp.ndarray = jnp.array(COMMUNICATION_PHASE),
        fixed_goal_position: jnp.ndarray = UNSET_POSITION,
        fixed_start_position: jnp.ndarray = UNSET_POSITION,
    ):
        timestep = self._env.reset(key)
        timestep = self._apply_curriculum_positions(
            timestep,
            fixed_goal_position=fixed_goal_position,
            fixed_start_position=fixed_start_position,
        )

        distance = self._goal_distance(timestep.state)

        def cond_fn(carry):
            _, _, goal_distance = carry
            return goal_distance < self.min_start_distance

        def body_fn(carry):
            rng, _, _ = carry
            rng, sample_key = jax.random.split(rng)
            sampled_timestep = self._env.reset(sample_key)
            sampled_timestep = self._apply_curriculum_positions(
                sampled_timestep,
                fixed_goal_position=fixed_goal_position,
                fixed_start_position=fixed_start_position,
            )
            goal_distance = self._goal_distance(sampled_timestep.state)
            return rng, sampled_timestep, goal_distance

        _, timestep, _ = jax.lax.while_loop(
            cond_fn,
            body_fn,
            (key, timestep, distance),
        )
        obs = self._split_observations(timestep, vision_radius, control_mode)
        return obs, timestep

    def reset_batch(
        self,
        keys: jnp.ndarray,
        vision_radius: jnp.ndarray = jnp.array(3.0),
        control_mode: jnp.ndarray = jnp.array(COMMUNICATION_PHASE),
        fixed_goal_position: jnp.ndarray = UNSET_POSITION,
        fixed_start_position: jnp.ndarray = UNSET_POSITION,
    ):
        return jax.vmap(self.reset, in_axes=(0, None, None, None, None))(
            keys,
            vision_radius,
            control_mode,
            fixed_goal_position,
            fixed_start_position,
        )

    def step(
        self,
        key: jnp.ndarray,
        timestep,
        action: jnp.ndarray,
        vision_radius: jnp.ndarray = jnp.array(3.0),
        control_mode: jnp.ndarray = jnp.array(COMMUNICATION_PHASE),
        fixed_goal_position: jnp.ndarray = UNSET_POSITION,
        fixed_start_position: jnp.ndarray = UNSET_POSITION,
    ):
        def reset_branch(_):
            reset_obs, reset_timestep = self.reset(
                key,
                vision_radius=vision_radius,
                control_mode=control_mode,
                fixed_goal_position=fixed_goal_position,
                fixed_start_position=fixed_start_position,
            )
            reward = jnp.asarray(0.0, dtype=jnp.float32)
            done = jnp.asarray(False)
            info = {
                "return": reset_timestep.info.get("return", reward),
                "task_reward": reward,
                "progress_reward": reward,
                "step_penalty": reward,
                "bump_penalty": reward,
                "goal_distance": self._goal_distance(reset_timestep.state),
            }
            return reset_obs, reset_timestep, reward, done, info

        def step_branch(_):
            old_distance = self._goal_distance(timestep.state)
            old_position = timestep.state.get_player().position
            next_timestep = self._env.step(timestep, action)
            new_distance = self._goal_distance(next_timestep.state)
            new_position = next_timestep.state.get_player().position
            obs = self._split_observations(next_timestep, vision_radius, control_mode)
            task_reward = next_timestep.reward.astype(jnp.float32)
            progress_reward = (old_distance - new_distance) * self.progress_reward_scale
            step_penalty = self.step_penalty
            bumped = jnp.logical_and(
                action == 2,
                jnp.all(new_position == old_position),
            )
            bump_penalty = jnp.where(
                bumped,
                self.bump_penalty,
                jnp.asarray(0.0, dtype=jnp.float32),
            )
            reward = task_reward + progress_reward - step_penalty - bump_penalty
            done = next_timestep.is_done()
            info = dict(next_timestep.info)
            info["task_reward"] = task_reward
            info["progress_reward"] = progress_reward
            info["step_penalty"] = step_penalty
            info["bump_penalty"] = bump_penalty
            info["goal_distance"] = new_distance
            return obs, next_timestep, reward, done, info

        return jax.lax.cond(timestep.is_done(), reset_branch, step_branch, operand=None)

    def step_batch(
        self,
        keys: jnp.ndarray,
        timesteps,
        actions: jnp.ndarray,
        vision_radius: jnp.ndarray = jnp.array(3.0),
        control_mode: jnp.ndarray = jnp.array(COMMUNICATION_PHASE),
        fixed_goal_position: jnp.ndarray = UNSET_POSITION,
        fixed_start_position: jnp.ndarray = UNSET_POSITION,
    ):
        return jax.vmap(self.step, in_axes=(0, 0, 0, None, None, None, None))(
            keys,
            timesteps,
            actions,
            vision_radius,
            control_mode,
            fixed_goal_position,
            fixed_start_position,
        )

    def render(
        self,
        timestep,
        control_mode: int = COMMUNICATION_PHASE,
    ):
        frame = np.asarray(observations.rgb(timestep.state)).copy()
        grid = np.asarray(observations.symbolic(timestep.state))
        player = np.asarray(timestep.state.get_player().position).astype(np.int32)
        tile_h = max(frame.shape[0] // grid.shape[0], 1)
        tile_w = max(frame.shape[1] // grid.shape[1], 1)
        row, col = int(player[0]), int(player[1])
        y0 = row * tile_h + tile_h // 4
        y1 = min((row + 1) * tile_h - tile_h // 4, frame.shape[0])
        x0 = col * tile_w + tile_w // 4
        x1 = min((col + 1) * tile_w - tile_w // 4, frame.shape[1])
        color = (
            np.array([32, 96, 224], dtype=np.uint8)
            if control_mode == self.SEER_NAV_PHASE
            else np.array([0, 0, 0], dtype=np.uint8)
        )
        frame[y0:y1, x0:x1] = color
        return frame


In [5]:
@functools.partial(jax.jit, static_argnames=("doer_apply_fn",))
def compute_cic(
    doer_apply_fn: Callable,
    doer_params: dict,
    transition_batch, 
    init_doer_carry: Tuple[jnp.ndarray, jnp.ndarray],
    rng: jax.random.PRNGKey
) -> jnp.ndarray:
    """
    Computes Causal Influence of Communication (CIC) by ablating the Seer's message
    and measuring the divergence in the Doer's resulting deterministic actions.
    """
    
    def scan_fn(carry, step_data):
        doer_carry = carry
        msg, loc, prop = step_data
        
        next_doer_carry, logits = doer_apply_fn(
            {"params": doer_params},
            doer_carry,
            loc,
            prop,
            msg
        )
        return next_doer_carry, logits

    # Transition batch shapes: (seq_len, num_envs, ...) from loop.py
    loc = transition_batch.local_obs
    prop = transition_batch.proprioception
    msg_true = transition_batch.message

    # True forward pass
    _, true_logits = jax.lax.scan(scan_fn, init_doer_carry, (msg_true, loc, prop))
    
    # Ablated forward pass: shuffle messages independently over time for each env.
    env_keys = jax.random.split(rng, msg_true.shape[1])
    msg_shuffled = jax.vmap(
        lambda key, msgs: jax.random.permutation(key, msgs, axis=0)
    )(env_keys, jnp.swapaxes(msg_true, 0, 1))
    msg_shuffled = jnp.swapaxes(msg_shuffled, 0, 1)
    
    _, ablated_logits = jax.lax.scan(scan_fn, init_doer_carry, (msg_shuffled, loc, prop))
    
    # Calculate CIC: divergence in deterministic policy
    true_actions = jnp.argmax(true_logits, axis=-1)
    ablated_actions = jnp.argmax(ablated_logits, axis=-1)
    
    cic = jnp.mean((true_actions != ablated_actions).astype(jnp.float32))
    
    return cic


In [6]:
def visualize_episode(
    env,
    params,
    rng,
    seer,
    doer,
    filename="episode.gif",
    vision_radius=jnp.array(2.0),
    max_steps=200,
    control_mode=jnp.array(1, dtype=jnp.int32),
    fixed_goal_position=UNSET_POSITION,
    fixed_start_position=UNSET_POSITION,
):
    """Run one greedy evaluation episode and save it as a GIF."""
    frames = []
    output_path = Path(filename)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    rng, reset_rng = jax.random.split(rng)
    obs, state = env.reset(
        reset_rng,
        vision_radius=vision_radius,
        control_mode=control_mode,
        fixed_goal_position=fixed_goal_position,
        fixed_start_position=fixed_start_position,
    )

    seer_carry = seer.initialize_carry(batch_size=1, hidden_size=128)
    doer_carry = doer.initialize_carry(batch_size=1, hidden_size=128)

    done = False
    solved = False
    step_count = 0

    while not bool(done) and step_count < max_steps:
        frame = env.render(state, control_mode=int(control_mode))
        frames.append(Image.fromarray(frame))

        global_map = obs["global_map"][None, ...]
        symbolic_state = obs["symbolic_state"][None, ...]
        local_view = obs["local_view"][None, ...]
        proprioception = obs["proprioception"][None, ...]

        seer_carry, message, _, seer_nav_logits = seer.apply(
            {"params": params["seer"]},
            seer_carry,
            global_map,
            symbolic_state,
        )

        if int(control_mode) == env.SEER_NAV_PHASE:
            action = jnp.argmax(seer_nav_logits[0]).astype(jnp.int32)
        else:
            doer_carry, action_logits = doer.apply(
                {"params": params["doer"]},
                doer_carry,
                local_view,
                proprioception,
                message,
            )
            action = jnp.argmax(action_logits[0]).astype(jnp.int32)

        rng, step_rng = jax.random.split(rng)
        obs, state, reward, done, info = env.step(
            step_rng,
            state,
            action,
            vision_radius=vision_radius,
            control_mode=control_mode,
            fixed_goal_position=fixed_goal_position,
            fixed_start_position=fixed_start_position,
        )
        solved = solved or bool(done) and float(info["task_reward"]) > 0.0

        step_count += 1

    if not frames:
        raise RuntimeError("Visualization episode produced no frames.")

    frames[0].save(
        output_path,
        save_all=True,
        append_images=frames[1:],
        duration=120,
        loop=0,
    )
    print(f"Episode saved to {output_path}")
    return output_path, solved


In [7]:
class Doer(nn.Module):
    """
    The 'Thief' or Motor Cortex network.
    Operates on local observations and executes commands via discrete actions.
    """
    fsq_levels: Sequence[int]
    num_actions: int = 6 # e.g., Move N/S/E/W, Toggle, Pick Up
    lstm_features: int = 128
    embed_dim: int = 16

    @nn.compact
    def __call__(
        self,
        carry: Tuple[jnp.ndarray, jnp.ndarray],
        local_obs: jnp.ndarray,
        proprioception: jnp.ndarray,
        message: jnp.ndarray
    ) -> Tuple[Tuple[jnp.ndarray, jnp.ndarray], jnp.ndarray]:
        """
        Args:
            carry: A tuple of (hidden_state, cell_state) for the LSTM.
            local_obs: Egocentric 3x3 grid view. Expected shape (batch, 3, 3, C) or zeros.
            proprioception: Internal states (e.g., carrying item). Expected shape (batch, features).
            message: The quantized $m_t$ vector from the Seer. Expected shape (batch, d).
            
        Returns:
            new_carry: Updated LSTM state for the next timestep $t+1$.
            action_logits: Unnormalized log probabilities for the discrete action space.
        """
        
        # 1. Message Encoder: Learned Lookup Table
        # To preserve gradients, we MUST NOT cast to int and use nn.Embed directly 
        # Instead, FSQ naturally provides continuous coordinates (that happen to be quantized).
        # We can just linearly project this entire vector directly into a latent space!
        message_features = nn.Dense(features=self.embed_dim * len(self.fsq_levels))(message)
        message_features = nn.relu(message_features)
        
        # 2. Local Visual Encoder
        # Even for a small 3x3 grid, a single convolution or a dense layer extracts features
        x = nn.Conv(features=16, kernel_size=(2, 2))(local_obs)
        x = nn.relu(x)
        x_flat = x.reshape((x.shape[0], -1))
        
        # 3. Proprioception Encoder
        p = nn.Dense(features=16)(proprioception)
        p = nn.relu(p)
        
        # 4. Fusion
        # Combine local vision, proprioception, and the embedded command
        fused_features = jnp.concatenate([x_flat, p, message_features], axis=-1)
        
        # 5. Reasoning Module: LSTM
        # Critical for integrating sequences of commands (e.g., maintaining a "Wait" state)
        lstm_cell = nn.LSTMCell(features=self.lstm_features)
        new_carry, lstm_out = lstm_cell(carry, fused_features)
        
        # 6. Output Head: Discrete Action Space
        # Projects the LSTM memory state into logits for physical actions
        action_logits = nn.Dense(features=self.num_actions)(lstm_out)
        
        return new_carry, action_logits

    @staticmethod
    def initialize_carry(batch_size: int, hidden_size: int) -> Tuple[jnp.ndarray, jnp.ndarray]:
        """Utility to generate the initial zero-state for the LSTM at the start of an episode."""
        return (
            jnp.zeros((batch_size, hidden_size)), 
            jnp.zeros((batch_size, hidden_size))
        )

In [8]:
class FSQ(nn.Module):
    """
    Finite Scalar Quantization (FSQ) module.
    Projects a continuous vector into a discrete hypercube.
    """
    # A sequence of integers defining the number of levels (L) per dimension (d).
    # e.g., levels=[5, 5, 5] means d=3 dimensions, each with 5 discrete levels.
    levels: Sequence[int]

    @nn.compact
    def __call__(self, z: jnp.ndarray) -> jnp.ndarray:
        """
        Args:
            z: The continuous "thought vector" from the Seer's reasoning module.
               Expected shape: (batch_size, ..., d) where d == len(self.levels)
        Returns:
            z_ste: The quantized vector with gradients preserved via STE.
        """
        levels = jnp.asarray(self.levels, dtype=z.dtype)
        
        # 1. Bounding: Restrict the unbounded input z to the range [-1, 1].
        # We use tanh as a standard, proven method for this projection.
        z_bound = jnp.tanh(z)
        
        # 2. Scaling: Map the [-1, 1] range to the grid [0, L - 1].
        # E.g., for L=5, this maps [-1, 1] to [0, 4].
        half_width = (levels - 1) / 2.0
        z_scaled = z_bound * half_width + half_width
        
        # 3. Quantization: Snap to the nearest integer grid point.
        if self.has_rng('noise'):
            # Inject uniform noise during training to "shake" the FSQ and prevent early mode collapse
            noise = jax.random.uniform(self.make_rng('noise'), z_scaled.shape, minval=-0.2, maxval=0.2)
            z_quantized = jnp.round(z_scaled + noise)
        else:
            z_quantized = jnp.round(z_scaled)
        
        # 4. Straight-Through Estimator (STE) Trick in JAX
        # Forward pass: Evaluates to z_quantized.
        # Backward pass: jax.lax.stop_gradient blocks the gradient from the 
        # non-differentiable z_quantized, so the gradient flows directly through z_scaled.
        z_ste = z_scaled + jax.lax.stop_gradient(z_quantized - z_scaled)
        
        return z_ste

In [9]:
class Seer(nn.Module):
    """
    The 'Hacker' or Prefrontal Cortex network.
    Observes the global state and generates a discrete compositional message.
    """
    fsq_levels: Sequence[int]
    num_actions: int = 3
    lstm_features: int = 128

    @nn.compact
    def __call__(
        self, 
        carry: Tuple[jnp.ndarray, jnp.ndarray], 
        map_obs: jnp.ndarray, 
        symbolic_obs: jnp.ndarray
    ) -> Tuple[Tuple[jnp.ndarray, jnp.ndarray], jnp.ndarray, jnp.ndarray, jnp.ndarray]:
        """
        Args:
            carry: A tuple of (hidden_state, cell_state) for the LSTM.
            map_obs: The Global Map View grid. Expected shape (batch, H, W, C).
            symbolic_obs: Guard Schedule + Sensor States. Expected shape (batch, features).
            
        Returns:
            new_carry: Updated LSTM state for the next timestep $t+1$.
            discrete_message: The quantized $m_t$ vector sent to the Doer.
            thought_vector: The continuous pre-quantization vector (useful for logging/critic).
            navigation_logits: Action logits used while the Seer is embodied.
        """
        
        # 1. Visual Encoder: CNN for the grid visual 
        x = nn.Conv(features=32, kernel_size=(3, 3), strides=(1, 1), padding='SAME')(map_obs)
        x = nn.relu(x)
        x = nn.Conv(features=64, kernel_size=(3, 3), strides=(1, 1), padding='SAME')(x)
        x = nn.relu(x)
        
        # Flatten visual features to a 1D vector per batch item
        # shape changes from (batch, H, W, channels) to (batch, H * W * channels)
        x_flat = x.reshape((x.shape[0], -1)) 
        
        # 2. Symbolic Encoder: MLP for symbolic data 
        y = nn.Dense(features=64)(symbolic_obs)
        y = nn.relu(y)
        
        # 3. Fusion
        # Concatenate the visual and symbolic pathways into a single representation 
        fused_features = jnp.concatenate([x_flat, y], axis=-1)
        
        # 4. Reasoning Module: LSTM 
        # Evaluates the current state in the context of the previous timestep [cite: 135]
        lstm_cell = nn.LSTMCell(features=self.lstm_features)
        new_carry, lstm_out = lstm_cell(carry, fused_features)
        
        # 5. Continuous Projection
        # Project LSTM hidden state to continuous vector z of size d.
        # Use Orthogonal init with higher scale to prevent FSQ mode collapse.
        thought_vector = nn.Dense(
            features=len(self.fsq_levels),
            kernel_init=nn.initializers.orthogonal(scale=2.0)
        )(lstm_out)
        
        # 6. Output Head: FSQ Discretizer 
        # Transforms the continuous thought vector into the discrete message $m_t$ 
        fsq = FSQ(levels=self.fsq_levels)
        discrete_message = fsq(thought_vector)

        # During the pretraining phase the Seer physically navigates the grid.
        navigation_logits = nn.Dense(features=self.num_actions)(lstm_out)

        return new_carry, discrete_message, thought_vector, navigation_logits

    @staticmethod
    def initialize_carry(batch_size: int, hidden_size: int) -> Tuple[jnp.ndarray, jnp.ndarray]:
        """Utility to generate the initial zero-state for the LSTM at the start of an episode."""
        return (
            jnp.zeros((batch_size, hidden_size)), 
            jnp.zeros((batch_size, hidden_size))
        )


In [10]:
def compute_gae(
    rewards: jnp.ndarray,
    values: jnp.ndarray,
    dones: jnp.ndarray,
    last_val: jnp.ndarray,
    gamma: float = 0.99,
    gae_lambda: float = 0.95
) -> Tuple[jnp.ndarray, jnp.ndarray]:
    """
    Computes Generalized Advantage Estimation (GAE) and target returns.
    
    Args:
        rewards: Array of rewards collected during the rollout. Shape: (num_steps,)
        values: Array of value predictions from the Critic. Shape: (num_steps,)
        dones: Array of boolean/integer done flags. Shape: (num_steps,)
        last_val: The Critic's value prediction for the state *after* the final step.
        gamma: Discount factor.
        gae_lambda: Bias-variance tradeoff parameter for GAE.
        
    Returns:
        advantages: The calculated GAE advantages. Shape: (num_steps,)
        returns: The target values for the Critic to learn from (Advantages + Values).
    """
    
    def _gae_step(carry, transition_data):
        """A single step of the reverse scan."""
        gae_t_plus_1 = carry
        reward, value, done, next_value = transition_data
        
        # Calculate the Temporal Difference (TD) error
        # If the episode ended (done=1), the next state has no value.
        delta = reward + gamma * next_value * (1.0 - done) - value
        
        # Calculate the advantage for the current timestep
        gae_t = delta + gamma * gae_lambda * (1.0 - done) * gae_t_plus_1
        
        # Pass the current advantage back as the carry for the previous timestep
        return gae_t, gae_t

    # To calculate the TD error, we need the value of the next state for every step.
    # We create an array of "next values" by shifting the values array by one 
    # and appending the bootstrap 'last_val' at the end.
    next_values = jnp.append(values[1:], last_val)
    
    # Pack the data for the scan
    scan_data = (rewards, values, dones, next_values)
    
    # Initialize the carry with 0.0 (the advantage after the final step)
    initial_gae = 0.0
    
    # Run the scan in reverse to propagate advantages backwards
    _, advantages = jax.lax.scan(_gae_step, initial_gae, scan_data, reverse=True)
    
    # The return value (target for the critic) is simply the advantage + the predicted value
    returns = advantages + values
    
    return advantages, returns

In [11]:
def make_rollout_step(
    env,
    seer_apply_fn,
    doer_apply_fn,
    critic_apply_fn,
    follow_reward_scale=0.1,
):
    """
    A closure that returns the JAX-compilable step function.
    Passing the environment and apply functions here avoids passing them 
    repeatedly into the compiled loop.
    """
    follow_reward_scale = jnp.asarray(follow_reward_scale, dtype=jnp.float32)
    
    def rollout_step(runner_state: Tuple, _):
        """
        Executes a single environment step and network forward pass.
        Designed to be passed directly to jax.lax.scan.
        """
        # Unpack the runner state
        (
            params,
            seer_carry,
            doer_carry,
            env_state,
            env_obs,
            rng,
            vision_radius,
            control_mode,
            fixed_goal_position,
            fixed_start_position,
        ) = runner_state
        num_envs = env_obs["global_map"].shape[0]
        communication_mode = control_mode == 1
        
        # Split the PRNG key for the stochastic actions
        rng, seer_rng, doer_rng, env_rng = jax.random.split(rng, 4)
        env_step_keys = jax.random.split(env_rng, num_envs)
        
        # 1. Seer Forward Pass (Prefrontal Cortex)
        # Enforcing CTDE: Seer gets the global view[cite: 131, 132].
        # In a custom jaxmarl wrapper, you would extract these from env_obs
        global_map = env_obs["global_map"]
        symbolic_obs = env_obs["symbolic_state"]
        
        next_seer_carry, discrete_message, _, seer_nav_logits = seer_apply_fn(
            {"params": params["seer"]}, 
            seer_carry, 
            global_map, 
            symbolic_obs
        )
        
        # 2. Doer Forward Pass (Motor Cortex)
        # Enforcing Functional Asymmetry: Doer gets local view and the message[cite: 137, 138].
        local_obs = env_obs["local_view"]
        proprioception = env_obs["proprioception"]
        
        next_doer_carry, action_logits = doer_apply_fn(
            {"params": params["doer"]}, 
            doer_carry, 
            local_obs, 
            proprioception, 
            discrete_message
        )
        _, null_action_logits = doer_apply_fn(
            {"params": params["doer"]},
            doer_carry,
            local_obs,
            proprioception,
            jnp.zeros_like(discrete_message),
        )
        
        # 3. Action Selection
        seer_pi = distrax.Categorical(logits=seer_nav_logits)
        pi = distrax.Categorical(logits=action_logits)
        seer_action = seer_pi.sample(seed=seer_rng)
        doer_action = pi.sample(seed=doer_rng)
        seer_log_prob = seer_pi.log_prob(seer_action)
        doer_log_prob = pi.log_prob(doer_action)
        env_action = jnp.where(communication_mode, doer_action, seer_action)
        message_action = jnp.argmax(action_logits, axis=-1)
        null_message_action = jnp.argmax(null_action_logits, axis=-1)
        action_change_bonus = (
            message_action != null_message_action
        ).astype(jnp.float32) * follow_reward_scale
        
        # 4. Critic Forward Pass (Centralized Training)
        # The critic evaluates the global state to guide learning[cite: 111].
        value = critic_apply_fn({"params": params["critic"]}, global_map)
        
        # 5. Environment Step
        # Step the jaxmarl environment using the chosen action
        # Note: jaxmarl expects a dictionary of actions for multi-agent, 
        # adapt this based on your specific wrapper implementation.
        next_env_obs, next_env_state, reward, done, info = env.step_batch(
            env_step_keys,
            env_state,
            env_action,
            vision_radius=vision_radius,
            control_mode=control_mode,
            fixed_goal_position=fixed_goal_position,
            fixed_start_position=fixed_start_position,
        )

        task_reward = info["task_reward"]
        progress_reward = info["progress_reward"]
        step_penalty = info["step_penalty"]
        bump_penalty = info["bump_penalty"]
        useful_communication = jnp.logical_or(
            progress_reward > 0.0,
            task_reward > 0.0,
        )
        follow_reward = jnp.where(
            jnp.logical_and(communication_mode, useful_communication),
            action_change_bonus,
            jnp.asarray(0.0, dtype=jnp.float32),
        )
        shared_comm_reward = (
            task_reward + progress_reward + follow_reward - step_penalty - bump_penalty
        )
        seer_reward = jnp.where(
            communication_mode,
            shared_comm_reward,
            task_reward + progress_reward - step_penalty - bump_penalty,
        )
        doer_reward = jnp.where(
            communication_mode,
            shared_comm_reward,
            jnp.asarray(0.0, dtype=jnp.float32),
        )
        reward = jnp.stack([seer_reward, doer_reward], axis=-1)

        done_mask = done[:, None]
        next_seer_carry = jax.tree_util.tree_map(
            lambda x: jnp.where(done_mask, jnp.zeros_like(x), x),
            next_seer_carry,
        )
        next_doer_carry = jax.tree_util.tree_map(
            lambda x: jnp.where(done_mask, jnp.zeros_like(x), x),
            next_doer_carry,
        )
        
        # 6. Build the Transition
        transition = Transition(
            global_obs=global_map,
            symbolic_obs=symbolic_obs,
            local_obs=local_obs,
            proprioception=proprioception,
            message=discrete_message,
            doer_action=doer_action,
            doer_log_prob=doer_log_prob,
            seer_action=seer_action,
            seer_log_prob=seer_log_prob,
            value=value,
            reward=reward,
            task_reward=task_reward,
            progress_reward=progress_reward,
            follow_reward=follow_reward,
            cic_reward_component=jnp.zeros_like(task_reward),
            cic_score=jnp.zeros_like(task_reward),
            step_penalty_component=step_penalty,
            bump_penalty_component=bump_penalty,
            done=done,
            # Advantage and return will be calculated post-rollout using GAE
            advantage=jnp.zeros_like(reward), 
            return_val=jnp.zeros_like(reward)
        )
        
        # Repack the updated runner state
        next_runner_state = (
            params,
            next_seer_carry,
            next_doer_carry,
            next_env_state,
            next_env_obs,
            rng,
            vision_radius,
            control_mode,
            fixed_goal_position,
            fixed_start_position,
        )
        
        return next_runner_state, transition

    return rollout_step

import functools

@functools.partial(jax.jit, static_argnames=("num_steps", "step_fn", "critic_apply_fn", "doer_apply_fn"))
def generate_trajectory_and_gae(
    params, rng, env_obs, env_state, seer_carry, doer_carry, vision_radius: jnp.ndarray, control_mode: jnp.ndarray, fixed_goal_position: jnp.ndarray, fixed_start_position: jnp.ndarray, cic_coef: jnp.ndarray, num_steps: int,
    step_fn, critic_apply_fn, doer_apply_fn
):
    """
    Executes the full episode rollout and computes GAE in a single compiled pass.
    Note: We pass the pre-compiled step_fn and initial states directly here 
    for better JAX compilation efficiency.
    """
    initial_runner_state = (
        params,
        seer_carry,
        doer_carry,
        env_state,
        env_obs,
        rng,
        vision_radius,
        control_mode,
        fixed_goal_position,
        fixed_start_position,
    )
    
    # 1. Execute the scan loop to collect the raw trajectory
    final_runner_state, trajectory_batch = jax.lax.scan(
        step_fn, initial_runner_state, None, length=num_steps
    )
    
    # 2. Extract the final state for Critic bootstrapping
    # Unpack the final runner state to get the last env_obs
    _, _, _, _, final_env_obs, final_rng, _, _, _, _ = final_runner_state
    
    # Enforce CTDE: The critic evaluates the global map 
    final_global_map = final_env_obs["global_map"]
    
    # 3. Calculate the bootstrap value (last_val)
    # The critic evaluates the state *after* the final step
    last_val = critic_apply_fn({"params": params["critic"]}, final_global_map)
    
    communication_mode = control_mode == 1

    def add_cic_bonus(_):
        cic_score = compute_cic(
            doer_apply_fn,
            params["doer"],
            trajectory_batch,
            doer_carry,
            final_rng,
        )
        per_step_cic_reward = cic_coef * cic_score / jnp.asarray(num_steps, dtype=jnp.float32)
        cic_reward_component = jnp.full_like(trajectory_batch.task_reward, per_step_cic_reward)
        cic_score_component = jnp.full_like(trajectory_batch.task_reward, cic_score)
        reward_with_cic = trajectory_batch.reward + cic_reward_component[..., None]
        return reward_with_cic, cic_reward_component, cic_score_component

    def skip_cic_bonus(_):
        zeros = jnp.zeros_like(trajectory_batch.task_reward)
        return trajectory_batch.reward, zeros, zeros

    reward_with_cic, cic_reward_component, cic_score_component = jax.lax.cond(
        jnp.logical_and(communication_mode, cic_coef > 0.0),
        add_cic_bonus,
        skip_cic_bonus,
        operand=None,
    )
    
    # 5. Compute GAE
    # Note: If you scale up to multiple environments (num_envs > 1), you would wrap 
    # compute_gae with jax.vmap(compute_gae, in_axes=1, out_axes=1) to vectorize 
    # the advantage calculation across all environments simultaneously.
    advantages, returns = jax.vmap(
        jax.vmap(
            compute_gae,
            in_axes=(1, 1, 1, 0, None, None),
            out_axes=1,
        ),
        in_axes=(2, 2, None, 1, None, None),
        out_axes=2,
    )(
        reward_with_cic,
        trajectory_batch.value,
        trajectory_batch.done,
        last_val,
        0.99,
        0.95,
    )
    
    # 5. Update the trajectory batch
    # Using the .replace() method provided by chex/flax dataclasses
    trajectory_batch = trajectory_batch.replace(
        reward=reward_with_cic,
        cic_reward_component=cic_reward_component,
        cic_score=cic_score_component,
        advantage=advantages,
        return_val=returns
    )
    
    return final_runner_state, trajectory_batch


In [12]:
# For the sake of a complete script, here is a pragmatic, standard Critic
class GlobalCritic(nn.Module):
    """Evaluates the global state to guide learning (CTDE)."""
    @nn.compact
    def __call__(self, global_map: jnp.ndarray) -> jnp.ndarray:
        x = nn.Conv(features=32, kernel_size=(3, 3))(global_map)
        x = nn.relu(x)
        x = x.reshape((x.shape[0], -1))
        x = nn.Dense(features=128)(x)
        x = nn.relu(x)
        value = nn.Dense(features=2)(x)
        return value


def reset_curriculum_batch(
    env,
    rng,
    num_envs,
    vision_radius,
    control_mode,
    fixed_goal_position,
    fixed_start_position,
):
    rng, env_rng = jax.random.split(rng)
    reset_keys = jax.random.split(env_rng, num_envs)
    env_obs, env_state = env.reset_batch(
        reset_keys,
        vision_radius=vision_radius,
        control_mode=control_mode,
        fixed_goal_position=fixed_goal_position,
        fixed_start_position=fixed_start_position,
    )
    return rng, env_obs, env_state


def sample_curriculum_anchor(
    env,
    rng,
    vision_radius,
    control_mode,
    fixed_goal_position=UNSET_POSITION,
    exclude_start_position=UNSET_POSITION,
    max_attempts=512,
):
    for _ in range(max_attempts):
        rng, sample_rng = jax.random.split(rng)
        _, timestep = env.reset(
            sample_rng,
            vision_radius=vision_radius,
            control_mode=control_mode,
            fixed_goal_position=fixed_goal_position,
        )
        start_position = env.player_position(timestep)
        if jnp.any(exclude_start_position < 0) or not bool(
            jnp.all(start_position == exclude_start_position)
        ):
            return rng, env.goal_position(timestep), start_position
    raise RuntimeError(
        "Failed to sample a new curriculum start position. "
        "This curriculum requires an environment with random starts, "
        "for example 'Navix-Empty-Random-8x8-v0'."
    )


def collect_message_action_trace(
    env,
    params,
    rng,
    seer,
    doer,
    vision_radius,
    max_steps,
    control_mode,
    fixed_goal_position,
    fixed_start_position,
):
    action_labels = ("turn_left", "turn_right", "forward")
    rng, reset_rng = jax.random.split(rng)
    obs, state = env.reset(
        reset_rng,
        vision_radius=vision_radius,
        control_mode=control_mode,
        fixed_goal_position=fixed_goal_position,
        fixed_start_position=fixed_start_position,
    )

    seer_carry = seer.initialize_carry(batch_size=1, hidden_size=128)
    doer_carry = doer.initialize_carry(batch_size=1, hidden_size=128)
    trace_lines = []
    done = False
    step_count = 0

    while not bool(done) and step_count < max_steps:
        global_map = obs["global_map"][None, ...]
        symbolic_state = obs["symbolic_state"][None, ...]
        local_view = obs["local_view"][None, ...]
        proprioception = obs["proprioception"][None, ...]

        seer_carry, message, _, _ = seer.apply(
            {"params": params["seer"]},
            seer_carry,
            global_map,
            symbolic_state,
        )
        doer_carry, action_logits = doer.apply(
            {"params": params["doer"]},
            doer_carry,
            local_view,
            proprioception,
            message,
        )

        message_np = np.asarray(message[0]).round(3).tolist()
        action = int(jnp.argmax(action_logits[0]))
        action_label = action_labels[action] if action < len(action_labels) else str(action)
        player_position = np.asarray(env.player_position(state)).tolist()
        trace_lines.append(
            f"t={step_count:02d} pos={player_position} msg={message_np} action={action_label}"
        )

        rng, step_rng = jax.random.split(rng)
        obs, state, _, done, info = env.step(
            step_rng,
            state,
            jnp.asarray(action, dtype=jnp.int32),
            vision_radius=vision_radius,
            control_mode=control_mode,
            fixed_goal_position=fixed_goal_position,
            fixed_start_position=fixed_start_position,
        )
        step_count += 1

    final_status = "solved" if bool(done) and float(info["task_reward"]) > 0.0 else "stopped"
    trace_lines.append(f"end={final_status} steps={step_count}")
    return rng, trace_lines


def evaluate_greedy_episode(
    env,
    params,
    rng,
    seer,
    doer,
    vision_radius,
    max_steps,
    control_mode,
    fixed_goal_position,
    fixed_start_position,
):
    rng, reset_rng = jax.random.split(rng)
    obs, state = env.reset(
        reset_rng,
        vision_radius=vision_radius,
        control_mode=control_mode,
        fixed_goal_position=fixed_goal_position,
        fixed_start_position=fixed_start_position,
    )

    seer_carry = seer.initialize_carry(batch_size=1, hidden_size=128)
    doer_carry = doer.initialize_carry(batch_size=1, hidden_size=128)
    done = False
    step_count = 0
    solved = False

    while not bool(done) and step_count < max_steps:
        global_map = obs["global_map"][None, ...]
        symbolic_state = obs["symbolic_state"][None, ...]
        local_view = obs["local_view"][None, ...]
        proprioception = obs["proprioception"][None, ...]

        seer_carry, message, _, seer_nav_logits = seer.apply(
            {"params": params["seer"]},
            seer_carry,
            global_map,
            symbolic_state,
        )

        if int(control_mode) == env.SEER_NAV_PHASE:
            action = jnp.argmax(seer_nav_logits[0]).astype(jnp.int32)
        else:
            doer_carry, action_logits = doer.apply(
                {"params": params["doer"]},
                doer_carry,
                local_view,
                proprioception,
                message,
            )
            action = jnp.argmax(action_logits[0]).astype(jnp.int32)

        rng, step_rng = jax.random.split(rng)
        obs, state, _, done, info = env.step(
            step_rng,
            state,
            action,
            vision_radius=vision_radius,
            control_mode=control_mode,
            fixed_goal_position=fixed_goal_position,
            fixed_start_position=fixed_start_position,
        )
        solved = solved or bool(done) and float(info["task_reward"]) > 0.0
        step_count += 1

    return rng, solved


def flatten_message_codes(message_batch, fsq_levels):
    levels = np.asarray(fsq_levels, dtype=np.int32)
    multipliers = np.ones_like(levels)
    for idx in range(len(levels) - 2, -1, -1):
        multipliers[idx] = multipliers[idx + 1] * levels[idx + 1]

    messages = np.rint(np.asarray(message_batch)).astype(np.int32)
    messages = np.clip(messages, 0, levels - 1)
    return (messages * multipliers).sum(axis=-1).astype(np.int32).reshape(-1)


def compute_message_stats(message_batch, fsq_levels):
    message_codes = flatten_message_codes(message_batch, fsq_levels)
    num_codes = int(np.prod(np.asarray(fsq_levels, dtype=np.int32)))
    counts = np.bincount(message_codes, minlength=num_codes).astype(np.float32)
    total = max(int(counts.sum()), 1)
    probs = counts / float(total)
    nonzero_probs = probs[probs > 0.0]
    entropy = float(-(nonzero_probs * np.log(nonzero_probs)).sum()) if nonzero_probs.size else 0.0
    max_entropy = float(np.log(num_codes)) if num_codes > 1 else 0.0
    normalized_entropy = entropy / max_entropy if max_entropy > 0.0 else 0.0
    unique_codes = int((counts > 0).sum())
    return {
        "message_codes": message_codes,
        "message_code_probs": probs,
        "rollout_message_entropy": entropy,
        "rollout_message_entropy_normalized": normalized_entropy,
        "rollout_message_unique_codes": unique_codes,
        "rollout_message_num_codes": num_codes,
    }


def log_curriculum_visualization(
    env,
    params,
    rng,
    seer,
    doer,
    config,
    update,
    phase_label,
    vision_radius,
    control_mode,
    fixed_goal_position,
    fixed_start_position,
):
    viz_path = Path(config["visualize_dir"]) / f"{phase_label}_{update:05d}.gif"
    output_path, solved = visualize_episode(
        env,
        params,
        rng,
        seer,
        doer,
        filename=str(viz_path),
        vision_radius=vision_radius,
        max_steps=config["visualize_max_steps"],
        control_mode=control_mode,
        fixed_goal_position=fixed_goal_position,
        fixed_start_position=fixed_start_position,
    )
    wandb.log(
        {
            "curriculum_reset_episode": wandb.Video(str(output_path), format="gif"),
            "curriculum_reset_episode_solved": int(solved),
        },
        commit=False,
    )


def main():
    # 1. Configuration and Logging
    config = {
        "learning_rate": 3e-4,
        "num_envs": 16,
        "num_steps": 128,
        "total_timesteps": 10_000_000,
        "env_id": "Navix-Empty-Random-8x8-v0",
        "fsq_levels": [4], # Defines the categorical hypercube
        "seed": 42,
        "follow_reward_scale": 0.1,
        "progress_reward_scale": 0.2,
        "cic_coef": 0.01,
        "doer_perception_level": 2,
        "max_doer_perception_level": 3,
        "curriculum_success_streak": 3,
        "curriculum_eval_every": 25,
        "seer_required_start_positions": 5,
        "communication_start_positions_per_level": 5,
        "release_goal_after_max_level": True,
        "min_start_distance": 1.0,
        "step_penalty": 0.01,
        "bump_penalty": 0.1,
        "message_trace_every": 100,
        "visualize_max_steps": 30,
        "visualize_dir": "artifacts/episodes",
    }
    
    wandb.init(entity="eleftheriaklk-ucl", project="brian_test", config=config)

    print(f"backend: {jax.default_backend()}")
    print(f"devices: {jax.devices()}")
    
    # 2. PRNG Key Initialization
    # JAX requires explicit, rigorous management of randomness
    rng = jax.random.PRNGKey(config["seed"])
    rng, seer_init_rng, doer_init_rng, critic_init_rng, env_rng = jax.random.split(rng, 5)

    # 3. Environment Instantiation
    raw_env = nx.make(config["env_id"])
    env = NavixGridWrapper(
        raw_env,
        progress_reward_scale=config["progress_reward_scale"],
        min_start_distance=config["min_start_distance"],
        step_penalty=config["step_penalty"],
        bump_penalty=config["bump_penalty"],
        doer_perception_level=config["doer_perception_level"],
    )
    seer_nav_mode = jnp.array(env.SEER_NAV_PHASE, dtype=jnp.int32)
    communication_mode = jnp.array(env.COMMUNICATION_PHASE, dtype=jnp.int32)
    control_mode = seer_nav_mode

    # 4. Initial Environment Reset
    fixed_goal_position = UNSET_POSITION
    fixed_start_position = UNSET_POSITION
    rng, fixed_goal_position, fixed_start_position = sample_curriculum_anchor(
        env,
        rng,
        vision_radius=jnp.array(3.0),
        control_mode=control_mode,
    )
    env.doer_perception_level = config["doer_perception_level"]
    rng, env_obs, env_state = reset_curriculum_batch(
        env,
        rng,
        config["num_envs"],
        vision_radius=jnp.array(3.0),
        control_mode=control_mode,
        fixed_goal_position=fixed_goal_position,
        fixed_start_position=fixed_start_position,
    )

    # 5. Network Instantiation
    seer = Seer(fsq_levels=config["fsq_levels"])
    doer = Doer(fsq_levels=config["fsq_levels"], num_actions=env.num_actions)
    critic = GlobalCritic()

    # 6. Parameter Initialization (Dummy Forward Passes)
    # We must pass data of the correct shape to initialize the Flax parameters
    dummy_map = env_obs["global_map"][:1]
    dummy_sym = env_obs["symbolic_state"][:1]
    dummy_local = env_obs["local_view"][:1]
    dummy_prop = env_obs["proprioception"][:1]
    dummy_msg = jnp.zeros((1, len(config["fsq_levels"])))
    
    init_seer_carry = seer.initialize_carry(1, 128)
    init_doer_carry = doer.initialize_carry(1, 128)

    seer_params = seer.init(seer_init_rng, init_seer_carry, dummy_map, dummy_sym)["params"]
    doer_params = doer.init(doer_init_rng, init_doer_carry, dummy_local, dummy_prop, dummy_msg)["params"]
    critic_params = critic.init(critic_init_rng, dummy_map)["params"]

    seer_carry = seer.initialize_carry(config["num_envs"], 128)
    doer_carry = doer.initialize_carry(config["num_envs"], 128)

    # Group parameters for the execution loop
    params = {"seer": seer_params, "doer": doer_params, "critic": critic_params}

    # 6. Optimizer and TrainState Setup
    # Optax provides the gradient transformation tools
    tx = optax.chain(
        optax.clip_by_global_norm(0.5),
        optax.adam(learning_rate=config["learning_rate"], eps=1e-5)
    )

    # Since the Seer and Doer act cooperatively to generate the trajectory,
    # we can conceptually treat them as a single "Actor" policy for the optimizer.
    # In a more advanced setup, you might give them separate optimizers.
    actor_state = TrainState.create(
        apply_fn=None, # We use specific apply fns in the update step
        params={"seer": seer_params, "doer": doer_params},
        tx=tx
    )
    
    critic_state = TrainState.create(
        apply_fn=critic.apply,
        params=critic_params,
        tx=tx
    )

    step_fn = make_rollout_step(
        env,
        seer.apply,
        doer.apply,
        critic.apply,
        follow_reward_scale=config["follow_reward_scale"],
    )
    current_start_success_streak = 0
    seer_mastered_starts = 0
    communication_mastered_starts = 0
    goal_randomization_enabled = False

    # 8. The Main Training Loop
    num_updates = config["total_timesteps"] // (config["num_steps"] * config["num_envs"])
    
    print(
        "Starting training... "
        f"(doer_perception_level={config['doer_perception_level']})"
    )
    for update in range(num_updates):
        rng, rollout_rng = jax.random.split(rng)
        
        # Curriculums
        vision_radius = jnp.clip(3.0 - 2.0 * (update / 1000.0), 1.0, 3.0)
        seer_entropy_coef = jnp.clip(0.1 - 0.09 * (update / 1000.0), 0.01, 0.1)
        
        # A. Collect Trajectory
        init_seer_carry = seer_carry
        init_doer_carry = doer_carry
        
        final_runner_state, trajectory_batch = generate_trajectory_and_gae(
            params,
            rollout_rng,
            env_obs,
            env_state,
            seer_carry,
            doer_carry,
            vision_radius,
            control_mode,
            fixed_goal_position,
            fixed_start_position,
            jnp.array(config["cic_coef"], dtype=jnp.float32),
            config["num_steps"],
            step_fn, critic.apply, doer.apply
        )
        
        # Extract for next loop iteration
        params, seer_carry, doer_carry, env_state, env_obs, _, _, _, _, _ = final_runner_state
        
        # B. Generalized Advantage Estimation (GAE)
        # GAE is now calculated inside generate_trajectory_and_gae
        
        # C. Update Networks
        rng, actor_rng, critic_rng = jax.random.split(rng, 3)
        
        batched_trajectory = jax.tree_util.tree_map(
            lambda x: jnp.swapaxes(x, 0, 1),
            trajectory_batch
        )
        batched_seer_carry = init_seer_carry
        batched_doer_carry = init_doer_carry

        actor_state, actor_metrics = update_actor(
            actor_state, batched_trajectory, batched_seer_carry, batched_doer_carry, 
            seer.apply,
            doer.apply,
            actor_rng,
            control_mode,
            tuple(config["fsq_levels"]),
            seer_entropy_coef,
        )
        critic_state, critic_metrics = update_critic(
            critic_state, batched_trajectory, critic.apply, critic_rng
        )
        
        # Sync updated parameters back to the params dictionary for the next rollout
        params["seer"] = actor_state.params["seer"]
        params["doer"] = actor_state.params["doer"]
        params["critic"] = critic_state.params

        success_events = jnp.logical_and(
            trajectory_batch.done,
            trajectory_batch.task_reward > 0.0,
        )
        completed_episodes = trajectory_batch.done.astype(jnp.int32).sum()
        rollout_num_successes = success_events.astype(jnp.int32).sum()
        rollout_success_rate = jnp.where(
            completed_episodes > 0,
            rollout_num_successes.astype(jnp.float32)
            / completed_episodes.astype(jnp.float32),
            jnp.asarray(0.0, dtype=jnp.float32),
        )
        message_stats = compute_message_stats(trajectory_batch.message, config["fsq_levels"])
        cic_score = float(trajectory_batch.cic_score.mean())
        
        # D. Logging
        if update % 10 == 0:
            in_communication_phase = int(control_mode) == env.COMMUNICATION_PHASE
            if int(control_mode) == env.SEER_NAV_PHASE:
                phase_label = "seer_nav"
            elif goal_randomization_enabled:
                phase_label = "communication_random_full"
            else:
                phase_label = "communication_random_start"
            phase_indicators = {
                "phase_seer_nav": int(phase_label == "seer_nav"),
                "phase_communication_random_start": int(
                    phase_label == "communication_random_start"
                ),
                "phase_communication_random_full": int(
                    phase_label == "communication_random_full"
                ),
            }
            wandb.log({
                "phase_label": phase_label,
                "doer_perception_level": config["doer_perception_level"],
                "current_start_success_streak": current_start_success_streak,
                "rollout_success_rate": rollout_success_rate,
                "task_reward": trajectory_batch.task_reward.mean(),
                "progress_reward": trajectory_batch.progress_reward.mean(),
                "cic_score": cic_score,
                "rollout_message_entropy_normalized": message_stats["rollout_message_entropy_normalized"],
                "rollout_message_unique_codes": message_stats["rollout_message_unique_codes"],
                "critic_loss": critic_metrics.get("critic_loss", 0.0),
                "message_distribution": wandb.Histogram(
                    np.asarray(message_stats["message_codes"])
                ),
                **phase_indicators,
            })
            print(
                f"Update {update}/{num_updates} | "
                f"Phase: {phase_label} | "
                f"Level: {config['doer_perception_level']} | "
                f"Start: ({int(fixed_start_position[0])}, {int(fixed_start_position[1])}) | "
                f"Goal: ({int(fixed_goal_position[0])}, {int(fixed_goal_position[1])}) | "
                f"Streak: {current_start_success_streak} | "
                f"SuccessRate: {float(rollout_success_rate):.3f} | "
                f"Seer Reward: {trajectory_batch.reward[..., 0].mean():.3f} | "
                f"Doer Reward: {trajectory_batch.reward[..., 1].mean():.3f} | "
                f"Task: {trajectory_batch.task_reward.mean():.3f} | "
                f"Progress: {trajectory_batch.progress_reward.mean():.3f} | "
                f"Follow: {trajectory_batch.follow_reward.mean():.3f} | "
                f"MsgH: {message_stats['rollout_message_entropy_normalized']:.3f} | "
                f"MsgUsed: {message_stats['rollout_message_unique_codes']}/"
                f"{message_stats['rollout_message_num_codes']} | "
                f"Step: {trajectory_batch.step_penalty_component.mean():.3f} | "
                f"Bump: {trajectory_batch.bump_penalty_component.mean():.3f} | "
                f"Seer Grad: {actor_metrics.get('seer_grad_norm', 0.0):.4f} | "
                f"Doer Grad: {actor_metrics.get('doer_grad_norm', 0.0):.4f} | "
                f"CIC: {cic_score:.3f}"
            )
            
            if (
                in_communication_phase
                and update % config["message_trace_every"] == 0
            ):
                rng, trace_rng = jax.random.split(rng)
                rng, trace_lines = collect_message_action_trace(
                    env,
                    params,
                    trace_rng,
                    seer,
                    doer,
                    vision_radius,
                    config["visualize_max_steps"],
                    control_mode,
                    fixed_goal_position,
                    fixed_start_position,
                )
                print("Communication trace:")
                for line in trace_lines:
                    print(line)

        if update > 0 and update % config["curriculum_eval_every"] == 0:
            rng, greedy_solved = evaluate_greedy_episode(
                env,
                params,
                rng,
                seer,
                doer,
                vision_radius,
                config["visualize_max_steps"],
                control_mode,
                fixed_goal_position,
                fixed_start_position,
            )
            current_start_success_streak = current_start_success_streak + 1 if greedy_solved else 0

            if int(control_mode) == env.SEER_NAV_PHASE:
                if current_start_success_streak >= config["curriculum_success_streak"]:
                    current_start_success_streak = 0
                    seer_mastered_starts += 1

                    if seer_mastered_starts >= config["seer_required_start_positions"]:
                        control_mode = communication_mode
                        env.doer_perception_level = config["doer_perception_level"]
                        communication_mastered_starts = 0
                        print("")
                        print("=" * 72)
                        print("NEW PHASE: communication")
                        print("=" * 72)
                        print("")
                        print("Seer navigation mastered on five starts; switching to communication phase.")
                    else:
                        print(
                            f"Seer mastered start {seer_mastered_starts}/"
                            f"{config['seer_required_start_positions']}."
                        )

                    rng, _, fixed_start_position = sample_curriculum_anchor(
                        env,
                        rng,
                        vision_radius,
                        control_mode,
                        fixed_goal_position=fixed_goal_position,
                        exclude_start_position=fixed_start_position,
                    )
                    rng, env_obs, env_state = reset_curriculum_batch(
                        env,
                        rng,
                        config["num_envs"],
                        vision_radius,
                        control_mode,
                        fixed_goal_position,
                        fixed_start_position,
                    )
                    seer_carry = seer.initialize_carry(config["num_envs"], 128)
                    doer_carry = doer.initialize_carry(config["num_envs"], 128)
                    print("")
                    print("=" * 72)
                    print(f"NEW RANDOM POSITION: {tuple(np.asarray(fixed_start_position).tolist())}")
                    print("=" * 72)
                    print("")
                    rng, viz_rng = jax.random.split(rng)
                    log_curriculum_visualization(
                        env,
                        params,
                        viz_rng,
                        seer,
                        doer,
                        config,
                        update,
                        "seer_nav_reset",
                        vision_radius,
                        control_mode,
                        fixed_goal_position,
                        fixed_start_position,
                    )

            elif int(control_mode) == env.COMMUNICATION_PHASE:
                if current_start_success_streak >= config["curriculum_success_streak"]:
                    current_start_success_streak = 0
                    communication_mastered_starts += 1

                    if (
                        communication_mastered_starts
                        >= config["communication_start_positions_per_level"]
                    ):
                        communication_mastered_starts = 0
                        if config["doer_perception_level"] < config["max_doer_perception_level"]:
                            config["doer_perception_level"] += 1
                            env.doer_perception_level = config["doer_perception_level"]
                            print("")
                            print("=" * 72)
                            print(
                                "NEW DOER PERCEPTION LEVEL: "
                                f"{config['doer_perception_level']}"
                            )
                            print("=" * 72)
                            print("")
                            print(
                                f"Curriculum advanced to doer_perception_level="
                                f"{config['doer_perception_level']}"
                            )
                        else:
                            if (
                                config["release_goal_after_max_level"]
                                and not goal_randomization_enabled
                            ):
                                goal_randomization_enabled = True
                                fixed_goal_position = UNSET_POSITION
                                fixed_start_position = UNSET_POSITION
                                print("")
                                print("=" * 72)
                                print("NEW SUBCASE: communication_random_full")
                                print("=" * 72)
                                print("")
                                print(
                                    "Max doer perception level mastered; releasing fixed goal "
                                    "and continuing in fully random Empty-Random-8x8."
                                )
                            else:
                                print("Max doer perception level reached; continuing with new starts.")

                    if goal_randomization_enabled:
                        previous_goal_position = fixed_goal_position
                        rng, fixed_goal_position, fixed_start_position = sample_curriculum_anchor(
                            env,
                            rng,
                            vision_radius,
                            control_mode,
                        )
                        if not np.array_equal(
                            np.asarray(previous_goal_position),
                            np.asarray(fixed_goal_position),
                        ):
                            print("")
                            print("=" * 72)
                            print(f"NEW RANDOM GOAL: {tuple(np.asarray(fixed_goal_position).tolist())}")
                            print("=" * 72)
                            print("")
                    else:
                        rng, _, fixed_start_position = sample_curriculum_anchor(
                            env,
                            rng,
                            vision_radius,
                            control_mode,
                            fixed_goal_position=fixed_goal_position,
                            exclude_start_position=fixed_start_position,
                        )
                    rng, env_obs, env_state = reset_curriculum_batch(
                        env,
                        rng,
                        config["num_envs"],
                        vision_radius,
                        control_mode,
                        fixed_goal_position,
                        fixed_start_position,
                    )
                    seer_carry = seer.initialize_carry(config["num_envs"], 128)
                    doer_carry = doer.initialize_carry(config["num_envs"], 128)
                    print("")
                    print("=" * 72)
                    print(f"NEW RANDOM POSITION: {tuple(np.asarray(fixed_start_position).tolist())}")
                    print("=" * 72)
                    print("")
                    rng, viz_rng = jax.random.split(rng)
                    phase_label = (
                        "communication_random_full_reset"
                        if goal_randomization_enabled
                        else "communication_random_start_reset"
                    )
                    log_curriculum_visualization(
                        env,
                        params,
                        viz_rng,
                        seer,
                        doer,
                        config,
                        update,
                        phase_label,
                        vision_radius,
                        control_mode,
                        fixed_goal_position,
                        fixed_start_position,
                    )

if __name__ == "__main__":
    main()


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/bezin/.netrc.
wandb: Currently logged in as: bezinwoke (bezinwoke-university-college-london) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


backend: cpu
devices: [CpuDevice(id=0)]
Starting training... (doer_perception_level=2)


KeyboardInterrupt: 

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x71e1d0077c80>> (for post_run_cell), with arguments args (<ExecutionResult object at 71e1903dbe90, execution_count=12 error_before_exec=None error_in_exec= info=<ExecutionInfo object at 71e1903dbec0, raw_cell="# For the sake of a complete script, here is a pra.." transformed_cell="# For the sake of a complete script, here is a pra.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://wsl%2Bubuntu/home/bezin/OpenEndedness/v2_JaxMarl/train.ipynb#W0sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


ConnectionResetError: Connection lost